# AfriQBench Notebook 03 — Controlled Noisy Simulation\n\nThis notebook extends the TFIM benchmark from ideal finite-shot simulation to a transparent controlled noise model in Qiskit Aer. It keeps sampling uncertainty and noise-induced bias conceptually separate.\n\nWe compare\n\n$$E_{\\mathrm{exact}} \\rightarrow E_{\\mathrm{ideal}} \\rightarrow E_{\\mathrm{noisy}}.$$\n\nThe noise model is synthetic and intentionally simple: single-qubit depolarizing error, two-qubit depolarizing error, and symmetric readout error.

## 1. Environment\n\nInstall the quantum and notebook extras from the repository root:\n\n```bash\npython -m pip install -e ".[quantum,notebook]"\n```

In [ ]:
from pathlib import Path\nimport sys\n\nrepo_root = Path.cwd()\nif not (repo_root / 'src').exists():\n    repo_root = repo_root.parent\nsys.path.insert(0, str(repo_root / 'src'))\n\nimport json\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nfrom afriqbench.metrics import absolute_error, relative_error\nfrom afriqbench.models.tfim import tfim_hamiltonian\nfrom afriqbench.quantum.noise import estimate_tfim_energy_noisy_aer\nfrom afriqbench.quantum.qiskit_tfim import bind_ansatz, estimate_tfim_energy_aer\nfrom afriqbench.reference.exact import ground_state\n

## 2. Load the same benchmark state used in Notebook 02

In [ ]:
n_qubits = 4\nJ = 1.0\nh = 1.0\nshots = 20_000\nseed = 12345\n\nbaseline_path = repo_root / 'results' / 'ideal_quantum_n4_h1_baseline.json'\nbaseline = json.loads(baseline_path.read_text(encoding='utf-8'))\nparameters = np.asarray(baseline['ansatz']['parameters'], dtype=float)\n\ncircuit = bind_ansatz(n_qubits, parameters, reps=2)\nH = tfim_hamiltonian(n_qubits, J=J, h=h, periodic=False)\nexact_energy, _ = ground_state(H)\n\nprint(f'Exact reference: {exact_energy:.10f}')\n

## 3. Ideal finite-shot control\n\nThis control has sampling uncertainty but no synthetic device noise.

In [ ]:
ideal = estimate_tfim_energy_aer(\n    circuit,\n    n_qubits=n_qubits,\n    J=J,\n    h=h,\n    shots=shots,\n    seed=seed,\n)\n\nprint(f"Ideal finite-shot energy: {ideal['energy']:.8f} ± {ideal['uncertainty']:.8f}")\nprint(f"Absolute error vs exact:  {absolute_error(ideal['energy'], exact_energy):.8f}")\n

## 4. Controlled noise sweep\n\nThe sweep deliberately increases two-qubit error faster than single-qubit error because entangling operations are usually a major contributor to circuit degradation. These values are benchmark controls, not claims about any specific hardware platform.

In [ ]:
noise_levels = [\n    {'label': 'ideal control', 'p1': 0.0, 'p2': 0.0, 'preadout': 0.0},\n    {'label': 'low', 'p1': 0.0005, 'p2': 0.005, 'preadout': 0.005},\n    {'label': 'moderate', 'p1': 0.001, 'p2': 0.01, 'preadout': 0.01},\n    {'label': 'elevated', 'p1': 0.002, 'p2': 0.02, 'preadout': 0.02},\n]\n\nrows = []\nfull_results = {}\n\nfor level in noise_levels:\n    result = estimate_tfim_energy_noisy_aer(\n        circuit,\n        n_qubits=n_qubits,\n        J=J,\n        h=h,\n        shots=shots,\n        seed=seed,\n        single_qubit_error=level['p1'],\n        two_qubit_error=level['p2'],\n        readout_error=level['preadout'],\n    )\n\n    full_results[level['label']] = result\n    rows.append({\n        'noise_level': level['label'],\n        'single_qubit_error': level['p1'],\n        'two_qubit_error': level['p2'],\n        'readout_error': level['preadout'],\n        'energy': result['energy'],\n        'shot_uncertainty': result['uncertainty'],\n        'absolute_energy_error': absolute_error(result['energy'], exact_energy),\n        'relative_energy_error': relative_error(result['energy'], exact_energy),\n    })\n\ndf = pd.DataFrame(rows)\ndf\n

## 5. Energy degradation\n\nError bars show propagated finite-shot standard error. They do **not** include systematic bias caused by the noise model.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))\nax.errorbar(\n    df['noise_level'],\n    df['energy'],\n    yerr=df['shot_uncertainty'],\n    marker='o',\n    capsize=4,\n)\nax.axhline(exact_energy, linestyle='--', linewidth=1, label='Exact reference')\nax.set_ylabel('Estimated energy')\nax.set_title('TFIM energy under controlled noise')\nax.legend()\nfig.tight_layout()\nplt.show()\n

## 6. Application-level error

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))\nax.plot(df['noise_level'], df['absolute_energy_error'], marker='o')\nax.set_ylabel(r'$|E_{\\mathrm{estimate}}-E_{\\mathrm{exact}}|$')\nax.set_title('Absolute TFIM energy error vs controlled noise')\nfig.tight_layout()\nplt.show()\n

## 7. Inspect term-level degradation\n\nTerm-level records are retained because aggregate energy can hide which observables are most affected.

In [ ]:
moderate_terms = pd.DataFrame(full_results['moderate']['terms'])\nmoderate_terms[['term', 'coefficient', 'expectation', 'uncertainty']]\n

## 8. Save reproducible noisy-run outputs

In [ ]:
results_dir = repo_root / 'results'\nresults_dir.mkdir(exist_ok=True)\n\ncsv_path = results_dir / 'noise_sweep_n4_h1_run.csv'\njson_path = results_dir / 'noise_sweep_n4_h1_run.json'\n\ndf.to_csv(csv_path, index=False)\njson_path.write_text(\n    json.dumps({\n        'benchmark_id': 'tfim-noise-sweep-n4-h1-v0',\n        'exact_energy': exact_energy,\n        'shots_per_term': shots,\n        'seed': seed,\n        'runs': full_results,\n    }, indent=2),\n    encoding='utf-8',\n)\n\nprint(csv_path)\nprint(json_path)\n

## Interpretation and next step\n\nThis notebook establishes a controlled sensitivity study: the exact problem, variational state, shot budget, and error channels are all explicit. The next execution layer should compare this synthetic model with **device-derived Aer noise** or real provider hardware, while preserving the same TFIM workload and reporting schema.\n\nCurrent `metriq-gym` local execution uses Qiskit Aer and also supports local IBM-device noise models through live calibration data or cached fake backend snapshots. That makes provider/device-derived noise a natural next integration target for AfriQBench.